# Class–Axiom Generator (Project 3, Step 2)

For each ontology (BFO, IES, CCOM, QUDT, CCOT, TO) this notebook:

1. runs a **SPARQL** query over the local Turtle file that returns every non-deprecated named class, its label, and each logical axiom attached to it (`rdfs:subClassOf`, `owl:equivalentClass`, `owl:disjointWith`);
2. renders anonymous class expressions (restrictions, unions, intersections) into readable Manchester-style strings;
3. writes one row per class to `src/data/<abbr>-axioms.xlsx` with the columns `iri`, `label`, `axiom_count`, `axioms`.

Deprecated classes (`owl:deprecated true`) are excluded in the query itself.

In [1]:
from pathlib import Path
import pandas as pd
from rdflib import Graph, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, OWL

NB_DIR = Path.cwd().resolve()
SRC_DIR = NB_DIR.parent / "src" if (NB_DIR.parent / "src").exists() else NB_DIR / "src"
DATA_DIR = SRC_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("SRC_DIR :", SRC_DIR)
print("DATA_DIR:", DATA_DIR)

TTL = {
    "bfo":  SRC_DIR / "bfo-core.ttl",
    "ies":  SRC_DIR / "ies.ttl",
    "ccom": SRC_DIR / "ccom.ttl",
    "qudt": SRC_DIR / "qudt.ttl",
    "ccot": SRC_DIR / "ccot.ttl",
    "to":   SRC_DIR / "time.ttl",
}

SRC_DIR : C:\Users\luaya\UB_applied_Onto\Ontology-Tradecraft\projects\project-3\assignment\src
DATA_DIR: C:\Users\luaya\UB_applied_Onto\Ontology-Tradecraft\projects\project-3\assignment\src\data


## The SPARQL query

* `VALUES ?classType` accepts both `owl:Class` and `rdfs:Class` (IES declares its classes as `rdfs:Class`).
* Two `FILTER NOT EXISTS` blocks drop deprecated classes, which are flagged in two different ways: `owl:deprecated true` (OWL-Time's `time:Year`, `time:January`) and `rdf:type owl:DeprecatedClass` (QUDT's `SIUnit`, `BaseUnit`, and others).
* The `OPTIONAL` axiom block returns one solution per axiom; `?target` is either a named class or a blank node (an anonymous class expression), which is rendered in Python below.
* The label is restricted to English or untagged literals so each class gets one label.

In [2]:
CLASS_AXIOM_QUERY = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>

SELECT DISTINCT ?iri ?label ?rel ?target
WHERE {
  VALUES ?classType { owl:Class rdfs:Class }
  ?iri a ?classType .
  FILTER ( isIRI(?iri) )

  FILTER NOT EXISTS {
    ?iri owl:deprecated ?d .
    FILTER ( ?d = true || lcase(str(?d)) = "true" )
  }
  FILTER NOT EXISTS { ?iri a owl:DeprecatedClass }

  OPTIONAL {
    ?iri rdfs:label ?label .
    FILTER ( lang(?label) = "" || langMatches(lang(?label), "en") )
  }

  OPTIONAL {
    VALUES ?rel { rdfs:subClassOf owl:equivalentClass owl:disjointWith }
    ?iri ?rel ?target .
  }
}
ORDER BY ?iri
"""

## Rendering class expressions (Manchester syntax with labels)

In [3]:
REL_NAME = {
    RDFS.subClassOf: "SubClassOf",
    OWL.equivalentClass: "EquivalentTo",
    OWL.disjointWith: "DisjointWith",
}

def name_of(g: Graph, node) -> str:
    """Prefer an English/untagged rdfs:label; fall back to a qname."""
    labels = [o for o in g.objects(node, RDFS.label) if isinstance(o, Literal)]
    labels.sort(key=lambda l: 0 if (l.language or "").startswith("en") else (1 if l.language is None else 2))
    if labels:
        return f"'{labels[0]}'"
    try:
        return g.namespace_manager.normalizeUri(node)
    except Exception:
        return str(node)

def rdf_list(g: Graph, head):
    while head is not None and head != RDF.nil:
        yield g.value(head, RDF.first)
        head = g.value(head, RDF.rest)

def render(g: Graph, expr) -> str:
    if isinstance(expr, URIRef):
        return name_of(g, expr)
    if isinstance(expr, Literal):
        return repr(str(expr))
    if (expr, RDF.type, OWL.Restriction) in g:
        prop = render(g, g.value(expr, OWL.onProperty))
        for pred, word in ((OWL.someValuesFrom, "some"), (OWL.allValuesFrom, "only"), (OWL.hasValue, "value")):
            v = g.value(expr, pred)
            if v is not None:
                return f"{prop} {word} {render(g, v)}"
        on_cls = g.value(expr, OWL.onClass) or g.value(expr, OWL.onDataRange)
        for pred, word in ((OWL.qualifiedCardinality, "exactly"), (OWL.minQualifiedCardinality, "min"),
                           (OWL.maxQualifiedCardinality, "max"), (OWL.cardinality, "exactly"),
                           (OWL.minCardinality, "min"), (OWL.maxCardinality, "max")):
            n = g.value(expr, pred)
            if n is not None:
                tail = f" {render(g, on_cls)}" if on_cls is not None else ""
                return f"{prop} {word} {n}{tail}"
        return f"{prop} [restriction]"
    for pred, op in ((OWL.intersectionOf, " and "), (OWL.unionOf, " or ")):
        head = g.value(expr, pred)
        if head is not None:
            return "(" + op.join(render(g, m) for m in rdf_list(g, head)) + ")"
    comp = g.value(expr, OWL.complementOf)
    if comp is not None:
        return f"not {render(g, comp)}"
    one_of = g.value(expr, OWL.oneOf)
    if one_of is not None:
        return "{" + ", ".join(render(g, m) for m in rdf_list(g, one_of)) + "}"
    return "(anonymous)"

## Run the query for every ontology and save `<abbr>-axioms.xlsx`

In [4]:
def class_axioms(ttl_path: Path) -> pd.DataFrame:
    g = Graph()
    g.parse(ttl_path)
    rows = {}
    for r in g.query(CLASS_AXIOM_QUERY):
        iri = str(r.iri)
        entry = rows.setdefault(iri, {"iri": iri, "label": None, "axioms": set()})
        if r.label is not None and entry["label"] is None:
            entry["label"] = str(r.label)
        if r.rel is not None and r.target is not None:
            entry["axioms"].add(f"{REL_NAME[r.rel]}: {render(g, r.target)}")
    out = []
    for e in rows.values():
        axioms = sorted(e["axioms"])
        out.append({"iri": e["iri"], "label": e["label"],
                    "axiom_count": len(axioms), "axioms": " | ".join(axioms)})
    return pd.DataFrame(out, columns=["iri", "label", "axiom_count", "axioms"]).sort_values("label", na_position="last")

summary = []
for abbr, path in TTL.items():
    df = class_axioms(path)
    out = DATA_DIR / f"{abbr}-axioms.xlsx"
    df.to_excel(out, index=False)
    summary.append({"ontology": abbr, "classes": len(df), "axioms": int(df["axiom_count"].sum()),
                    "classes_without_axioms": int((df["axiom_count"] == 0).sum()), "file": out.name})
pd.DataFrame(summary)

,ontology,classes,axioms,classes_without_axioms,file
0,bfo,36,76,1,bfo-axioms.xlsx
1,ies,510,579,0,ies-axioms.xlsx
2,ccom,28,50,1,ccom-axioms.xlsx
3,qudt,167,237,0,qudt-axioms.xlsx
4,ccot,33,59,0,ccot-axioms.xlsx
5,to,18,62,3,to-axioms.xlsx


## Sanity check: no deprecated classes leaked into the exports

In [5]:
for abbr in TTL:
    df = pd.read_excel(DATA_DIR / f"{abbr}-axioms.xlsx")
    bad_label = df["label"].astype(str).str.contains("deprecated", case=False, na=False).sum()
    bad_axiom = df["axioms"].astype(str).str.contains(r"owl:deprecated", case=False, na=False).sum()
    print(f"{abbr:>4}: {len(df):>3} classes | deprecated labels: {bad_label} | deprecated axioms: {bad_axiom}")

 bfo:  36 classes | deprecated labels: 0 | deprecated axioms: 0
 ies: 510 classes | deprecated labels: 0 | deprecated axioms: 0
ccom:  28 classes | deprecated labels: 0 | deprecated axioms: 0
qudt: 167 classes | deprecated labels: 0 | deprecated axioms: 0
ccot:  33 classes | deprecated labels: 0 | deprecated axioms: 0
  to:  18 classes | deprecated labels: 0 | deprecated axioms: 0
